# PT01: Introduction to PyTorch

If you do not have prior exposure to PyTorch, read and run this notebook before PT02 (or at least go through the [basic documentation of PyTorch](https://docs.pytorch.org/docs/2.14/index.html)). Knowledge of NumPy helps, since most of the array syntax should look familiar.

We will use a small batch of examples to compute predictions, a loss, and its gradients, with additional material on the internals of PyTorch towards the end.

Before running a cell, try to predict its output or at least its shape. If something is unexpected, change a small example and run it again.

**Important**: We use `jaxtyping` to annotate shapes for the inputs and outputs when feasible. For example, `Float[torch.Tensor, "batch features"]` means a floating-point tensor with two named axes. If the import fails, install `jaxtyping` with `%pip install jaxtyping` in a code cell, then rerun the imports.

In [ ]:
%pip install jaxtyping

In [ ]:
import torch
from jaxtyping import Float, Int
from typing import Optional
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)
print("PyTorch", torch.__version__)

## 1. Tensors

The core array type is `torch.Tensor`. We can construct one with `torch.tensor`, or use functions such as `torch.zeros` and `torch.randn`. Here, each row is an example and each column is a feature: $X \sim (B,D)$, with $B=3$ and $D=2$.

In [ ]:
X = torch.tensor([[1., 2.], [3., 1.], [2., 4.]])
print(X)
print("Shape:", X.shape)
print("Dtype:", X.dtype)
print("Device:", X.device)

`X.to(...)` can be used to move the data to a different device, but we don't need it for now. 

Indexing follows NumPy's zero-based convention. Compare the last two expressions below: selecting a column can remove its dimension.

In [ ]:
print("Second example:", X[1])
print("First two examples:\n", X[:2])
print("Last feature:", X[:, -1], X[:, -1].shape)
print("Last feature, keeping its axis:\n", X[:, -1:], X[:, -1:].shape)

## 2. A batch of linear predictions

For one example, our model is $z=Wx+b$. With examples stored in rows, the batched version is

$$Z=XW^\top+b.$$

Here $W\sim(C,D)$ and $Z\sim(B,C)$. The bias has shape $(C,)$ and is broadcast over the examples. `@` is matrix multiplication; `*` is elementwise multiplication.

In [ ]:
W = torch.tensor([[1., -1.], [0.5, 2.]])  # (C, D)
b = torch.tensor([0.2, -0.3])             # (C,)
Z = X @ W.T + b                         # (B, C)
print(Z)
torch.testing.assert_close(Z[0], W @ X[0] + b)

Reductions remove an axis unless we ask to keep it (e.g., if the result should be used in another broadcast).

For example, subtracting the mean of each feature requires reducing over the examples, not over the features. Try changing `dim=0` to `dim=1` below and inspect what gets centered.

In [ ]:
feature_mean = X.mean(dim=0, keepdim=True)
centered = X - feature_mean
print("Mean shape:", feature_mean.shape)
print(centered)
torch.testing.assert_close(centered.mean(dim=0), torch.zeros(2), atol=1e-6, rtol=0)

## 3. Computing gradients

Consider a regression model $\hat y=Xw+b$ with a mean squared error. We tell PyTorch which arrays need gradients (the parameters) using `requires_grad=True`. Before running the next cells, predict the shapes of the loss, `w.grad` and `bias.grad`.

In [ ]:
y = torch.tensor([1., -1., 2.])
w = torch.tensor([0.2, -0.1], requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

prediction = X @ w + bias
loss = ((prediction - y) ** 2).mean()
print("Loss:", loss.item(), "Shape:", loss.shape)
print("Parameter is a leaf:", w.is_leaf)
print("Prediction is a leaf:", prediction.is_leaf)
loss.backward()
print("w.grad:", w.grad)
print("bias.grad:", bias.grad)

`backward()` computes gradients of this scalar loss. By default, `.grad` is populated for the leaf tensors that require gradients, such as `w` and `bias`. An intermediate tensor can require gradients without storing them in its own `.grad` field.

PyTorch records operations during the forward pass. Backward uses the saved information and normally releases the saved tensors afterward. To compute gradients for a new step, run a new forward pass. Reusing this same `loss` for another backward pass would require retaining its graph, which we do not need here.

## 4. Updating the parameters

Recall the update $\theta\leftarrow\theta-\eta\nabla L(\theta)$. The update of the parameters does not need gradients, so we perform it inside `torch.no_grad()` (this avoids storing intermediate quantities, as we will see when studying automatic differentiation). We then clear the gradients. `backward()` **adds** to existing gradients; it does not overwrite them.

In [ ]:
lr = 0.01
before = loss.item()
with torch.no_grad():
    w -= lr * w.grad
    bias -= lr * bias.grad
w.grad = None
bias.grad = None

after = ((X @ w + bias - y) ** 2).mean()
print(f"Loss before: {before:.4f}; after one update: {after.item():.4f}")
assert after.item() < before  # This example uses a sufficiently small step.

We can see accumulation without training anything. Keep the parameters fixed and differentiate the same function twice, rebuilding the forward computation each time.

In [ ]:
a = torch.tensor([1., 2.], requires_grad=True)
a.square().sum().backward()
first_gradient = a.grad.clone()
a.square().sum().backward()
print("First backward:", first_gradient)
print("Second backward:", a.grad)
torch.testing.assert_close(a.grad, 2 * first_gradient)
a.grad = None

For logging, `loss.item()` gives a Python number. `tensor.detach()` gives a tensor disconnected from its gradient history. Use these when storing results that will not be differentiated later.

`torch.autograd.grad` is another interface: it returns gradients directly instead of accumulating them into `.grad`. We will use it again when studying autodiff.

In [ ]:
a = torch.tensor([1., 2.], requires_grad=True)
(gradient,) = torch.autograd.grad(a.square().sum(), a)
print(gradient)
assert a.grad is None

## 5. Classification uses the same steps

For classification, $Z\sim(B,C)$ contains logits. The target $y\sim(B,)$ contains integer class indices. Softmax turns logits into probabilities when we need to inspect them.

PyTorch's cross-entropy function takes **logits directly**. It combines the logarithm and normalization in a numerically stable form. Do not pass the probabilities to it.

In [ ]:
logits = torch.tensor([[2., 0., -1.], [-1., 0., 2.]], requires_grad=True)
labels = torch.tensor([0, 2], dtype=torch.long)
classification_loss = F.cross_entropy(logits, labels)
classification_loss.backward()
print("Probabilities:\n", logits.detach().softmax(dim=-1))
print("Loss:", classification_loss.item())
print("Gradient shape:", logits.grad.shape)

For an ordinary training step we will clear gradients, compute predictions and a scalar loss, call `backward()`, then update the parameters. Evaluation needs predictions but does not need to build their gradient history.

`model.eval()` and `torch.no_grad()` do different things. The first changes the behavior of layers such as dropout and batch normalization. The second disables gradient recording. In PT02 we will use both when evaluating a model.

Before PT02, make sure you can explain the bias broadcast, the shapes of the gradients, and what happens if we forget to clear them. The remaining examples are optional.

## Before PT02: a short check

Try these without running code first. Then use the next cell to check your reasoning.

1. `A` has shape `(3, 2)`. We want to center each row. What goes wrong with `A - A.mean(dim=1)`? Repair it and give the shape of the row means.
2. `a = torch.tensor([1., 2.], requires_grad=True)`. We call `a.square().sum().backward()` twice, rebuilding the forward computation each time and keeping `a` fixed. What is `a.grad` after each call? Where should we clear it if we want a fresh gradient?
3. For `X @ W.T + b` with shapes `(5, 4)`, `(3, 4)` and `(3,)`, give the shapes of the logits, a mean cross-entropy loss, and the two parameter gradients.

If you cannot explain one of these, revisit the corresponding example before PT02.

In [ ]:
# Try your repairs and shape predictions here.

<details>
<summary>Check your reasoning</summary>

1. Reducing over `dim=1` gives `(3,)`, which cannot broadcast against `(3, 2)`. Use `A.mean(dim=1, keepdim=True)` to keep shape `(3, 1)`.
2. The gradients are `[2., 4.]` and `[4., 8.]`. Set `a.grad = None` before each backward call when you want only the current gradient. A new forward pass does not clear it.
3. Logits: `(5, 3)`; mean loss: `()`; weight gradient: `(3, 4)`; bias gradient: `(3,)`.

</details>

## Optional: views, copies and in-place operations

Some indexing operations return a view, while others return a copy, as shown below.

In [ ]:
A = torch.arange(6.).reshape(3, 2)
view = A[:2]
copy = A[[0, 1]]
view[0, 0] = -10
print("Original:\n", A)
print("Copy made before the update:\n", copy)
assert A[0, 0].item() == -10 and copy[0, 0].item() == 0

# An underscore usually marks an in-place operation.
A.add_(1)
print(A)

In-place changes deserve more care when autograd is involved: backward may need a value that the operation overwrites. The explicit parameter update above is a controlled use after backward.

For dense strided tensors, a transpose changes the strides without moving the underlying data. `reshape` returns a view when possible and otherwise copies. We will return to layout when it matters for performance.

In [ ]:
A = torch.arange(6.).reshape(3, 2)
print("A:", A.shape, A.stride(), A.is_contiguous())
print("A.T:", A.T.shape, A.T.stride(), A.T.is_contiguous())
print("Flattened transpose:", A.T.reshape(-1))

## Optional: a few array operations

This is a collection of short examples - you can try writing the function yourself before looking at the implementation. First, sum the main diagonal and the antidiagonal. For an odd-sized matrix, count the center once in each diagonal, so it contributes twice to the sum.

In [ ]:
def sum_diagonals(A: Float[torch.Tensor, "n n"]) -> Float[torch.Tensor, ""]:
    return A.diagonal().sum() + A.flip(dims=[1]).diagonal().sum()

A = torch.arange(1., 10.).reshape(3, 3)
torch.testing.assert_close(sum_diagonals(A), torch.tensor(30.))
print(sum_diagonals(A))

Next, normalize each row to have mean zero and variance one. Here variance means the average squared deviation (`correction=0`). A constant row becomes zero; there is no way to give it unit variance by rescaling its deviations.

In [ ]:
def normalize_rows(A: Float[torch.Tensor, "batch features"], eps: float = 1e-8) -> Float[torch.Tensor, "batch features"]:
    centered = A - A.mean(dim=1, keepdim=True)
    std = A.std(dim=1, correction=0, keepdim=True)
    return centered / std.clamp_min(eps)

A = torch.tensor([[1., 2., 4.], [4., 4., 4.]])
normalized = normalize_rows(A)
print(normalized)
torch.testing.assert_close(normalized.mean(dim=1), torch.zeros(2), atol=1e-6, rtol=0)
torch.testing.assert_close(normalized.var(dim=1, correction=0), torch.tensor([1., 0.]))

Finally, compute all pairwise cosine similarities. Normalize the rows and multiply by the transpose. This operation will return when we discuss attention-based layers.

Cosine similarity is undefined for a zero vector. For this implementation we assign zero to comparisons involving a zero vector, matching PyTorch's normalization convention below.

In [ ]:
def cosine_similarity(A: Float[torch.Tensor, "batch features"]) -> Float[torch.Tensor, "batch batch"]:
    unit = F.normalize(A, p=2, dim=1, eps=1e-12)
    return unit @ unit.T

A = torch.tensor([[1., 0.], [0., 1.], [-1., 0.], [0., 0.]])
expected = torch.tensor([[1., 0., -1., 0.], [0., 1., 0., 0.],
                         [-1., 0., 1., 0.], [0., 0., 0., 0.]])
torch.testing.assert_close(cosine_similarity(A), expected)
print(cosine_similarity(A))

## Further reading

- [PyTorch tensor tutorial](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
- [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)
- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)